# t-Distributed Stochastic Neighbor Embedding (t-SNE) — From Theory to Practice

## What This Notebook Covers
This notebook is the practical, interactive companion to the **t-SNE README**. We will explore how high-dimensional, non-linear manifolds can be compressed and visualized in a 2D space. You will learn to load high-dimensional digit image datasets, standard-scale and normalize inputs, apply PCA as an initial noise-filtering step, implement t-SNE from scratch using only NumPy, run optimized production-grade scikit-learn SVD/Barnes-Hut pipelines, tune critical hyperparameters like perplexity, and quantitatively evaluate embedding quality using the Trustworthiness metric and downstream KNN accuracy sweeps.

## What You Will Accomplish
- Describe the crowding problem in dimensionality reduction and how the Student t-distribution resolves it.
- Prepare high-dimensional image data (Standardization and PCA preprocessing) before neighbor embedding.
- Build a basic t-SNE solver from scratch using pure NumPy matrix multiplication to compute conditional and joint probability distributions.
- Apply Scikit-learn's optimized Barnes-Hut `TSNE` estimator to compress 5,000 samples of handwritten digit images from 784 dimensions to 2 dimensions.
- Visualize the resulting embeddings using scatter plots and contour density distributions to assess cluster separation.
- Run hyperparameter sweeps over perplexity values to observe the transition from fragmented local structures to merged global blobs.
- Evaluate embedding quality quantitatively using Scikit-learn's `trustworthiness` score and cross-validated KNN classifier accuracy.

## Before You Start (Prerequisites)
- Comfort manipulating Python loops, dictionaries, and NumPy array slices.
- Familiarity with basic linear algebra concepts (vectors, dot products, matrix multiplication `@`).
- Basic familiarity with PCA (dimensionality reduction) is helpful but not required.

## About the Dataset
We use the benchmark **MNIST Handwritten Digits dataset**, containing 70,000 grayscale images of handwritten digits (0–9), each of size 28×28 pixels. Each image is represented as a flat vector of **784 numbers** (pixel brightness values from 0 to 255). Our goal is to compress these 784 dimensions down to just 2 dimensions, keeping similar digits close together.

We load it using `sklearn.datasets.fetch_openml`.
---

## 1. Setup & Workspace Preparation

### WHY?
Importing all scientific computing, visualization, and validation libraries at the beginning of the notebook prevents path resolution errors and ensures all seeds are fixed for reproducibility.

### HOW?
We import NumPy, Pandas, Matplotlib, Seaborn, and key Scikit-learn modules, set formatting options, and fix numpy seed to 42.

In [ ]:
# Import NumPy for manual matrix operations and distance computations
import numpy as np

# Import Pandas for displaying dataframes and statistical summaries
import pandas as pd

# Import visualization libraries for plotting scatter and contour charts
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

# Import dataset loader, preprocessors, and dimensionality reduction tools from sklearn
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, classification_report

# Set utilities for timing code execution and suppressing warnings
import time
import warnings
warnings.filterwarnings('ignore')

# Set seaborn style for clean grids
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

# Fix numpy seed for reproducibility
np.random.seed(42)

print("All libraries imported and seed fixed to 42.")

## 2. Dataset Loading & Stratified Subsampling

### WHY?
Exact t-SNE has a time complexity of $O(N^2)$, meaning that running the algorithm on the full 70,000 samples of MNIST would be computationally prohibitive. We will take a stratified subsample of 5,000 samples (500 per digit class) to ensure balanced representation while keeping runtimes manageable.

### HOW?
We load MNIST using `fetch_openml`, select the subset proportionally using index slicing, and verify the class distribution using value counts.

In [ ]:
print("📥 Loading MNIST dataset from OpenML (this may take up to 60 seconds)...\n")

# Fetch MNIST dataset (as_frame=False to return numpy arrays directly)
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_raw = mnist.data.astype(np.float32)
y_raw = mnist.target.astype(int)

print(f"✅ Full dataset loaded! Shape: {X_raw.shape[0]} samples × {X_raw.shape[1]} features")

# Take a stratified subsample of 5,000 samples (500 per digit class)
samples_per_class = 500
selected_indices = []

for digit in range(10):
    digit_indices = np.where(y_raw == digit)[0]
    chosen_indices = np.random.choice(digit_indices, size=samples_per_class, replace=False)
    selected_indices.extend(chosen_indices)

selected_indices = np.array(selected_indices)
# Shuffle selected indices to remove ordering bias
np.random.shuffle(selected_indices)

X = X_raw[selected_indices]
y = y_raw[selected_indices]

print(f"\n📊 Subsampled dataset created! Target shape: {X.shape[0]} samples × {X.shape[1]} features")
print(f"   Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

## 3. Exploratory Data Analysis (EDA)

### WHY?
Before performing dimensionality reduction, it is crucial to visually inspect sample images to understand their raw features and make sure the data loaded correctly.

### HOW?
We plot a grid of 25 random handwritten digit images from our subset, with each image labeled and colored according to its true digit class.

In [ ]:
# Set up the figure grid
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
fig.suptitle("Sample MNIST Handwritten Digits from Our Subsample", fontsize=16, fontweight='bold', y=0.95)

colors = plt.cm.tab10(np.linspace(0, 1, 10))

for i, ax in enumerate(axes.flat):
    # Reshape the flat 784-D vector back into a 28x28 grayscale image grid
    digit_img = X[i].reshape(28, 28)
    true_label = y[i]
    
    # Plot image with coordinates off
    ax.imshow(digit_img, cmap='gray_r')
    ax.axis('off')
    
    # Add colored boundary box representing the digit class
    rect = plt.Rectangle((0,0), 27, 27, fill=False, color=colors[true_label], linewidth=3)
    ax.add_patch(rect)
    ax.set_title(f"Digit: {true_label}", color=colors[true_label], fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Preprocessing: Standardization and PCA Pre-processing

### WHY?
Dimensionality reduction models are highly sensitive to variable scales. Because pixel values range from 0 to 255, dividing by 255.0 normalizes them to $[0, 1]$, making distance calculations scale-invariant. 

We then run **PCA** to reduce the dimensionality from 784 dimensions to 50 dimensions. This pre-processing step filters out high-frequency noise, speeds up subsequent t-SNE distance computations, and prevents local optimization traps.

### HOW?
We divide the data by 255.0, fit `PCA(n_components=50)` on the normalized features, and plot a Scree plot showing cumulative variance.

In [ ]:
# Step 4a: Normalize pixel features to [0, 1] range
X_normalized = X / 255.0

# Step 4b: Apply PCA to reduce dimensionality to 50 components
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_normalized)

print(f"Original feature space dimension: {X_normalized.shape[1]} (pixels)")
print(f"Reduced feature space dimension : {X_pca.shape[1]} (principal components)")
print(f"Total variance retained in 50 components: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# Step 4c: Plot PCA Scree plot diagnostic dashboard
fig, ax = plt.subplots(figsize=(10, 5))
components = np.arange(1, 51)
ax.bar(components, pca.explained_variance_ratio_, alpha=0.7, color='#2196F3', label='Individual Variance')
ax.step(components, np.cumsum(pca.explained_variance_ratio_), where='mid', color='#FF9800', label='Cumulative Variance')
ax.axhline(y=0.85, color='r', linestyle='--', label='85% Variance Threshold')
ax.set_xlabel('Principal Component Index', fontsize=11)
ax.set_ylabel('Variance Explained Ratio', fontsize=11)
ax.set_title('PCA Scree Plot: Filtering Image Noise', fontsize=13, fontweight='bold')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 5. Mathematical Blueprint of t-SNE (Review)

t-SNE models similarity in the original high-dimensional space ($p_{j|i}$) using a **Gaussian distribution**:

$$p_{j|i} = \frac{\exp(-\lVert x_i - x_j \rVert^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\lVert x_i - x_k \rVert^2 / 2\sigma_i^2)}$$

To make the metrics robust to outliers, we calculate the symmetric joint probability:

$$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2N}$$

In the low-dimensional map, we compute Cauchy similarity $q_{ij}$ using a **Student t-distribution (1 DOF)** to resolve the crowding problem:

$$q_{ij} = \frac{(1 + \lVert y_i - y_j \rVert^2)^{-1}}{\sum_{k \neq l} (1 + \lVert y_k - y_l \rVert^2)^{-1}}$$

We minimize the difference between these two distributions using the **KL Divergence cost function**:

$$C = KL(P \parallel Q) = \sum_{i \neq j} p_{ij} \log\frac{p_{ij}}{q_{ij}}$$

---

## 6. Manual t-SNE Implementation (From Scratch using NumPy)

### WHY?
Implementing t-SNE from scratch using only NumPy helps you understand how high-dimensional similarities are calculated, how binary searches match perplexities, and how gradients minimize the cost function.

### HOW?
We implement the `SimpleTSNE` class containing functions for pairwise squared distances, Gaussian bandwidth binary search matching target perplexity, Cauchy joint probability calculation, KL divergence gradient calculation, and a gradient descent loop with momentum and early exaggeration.

In [ ]:
class SimpleTSNE:
    def __init__(self, n_components=2, perplexity=30.0, n_iter=300, learning_rate=100.0, random_state=42):
        self.n_components = n_components
        self.perplexity = perplexity
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.costs = []

    def _pairwise_squared_distances(self, X):
        # Compute squared Euclidean distances using the sum of squares trick
        # ||a-b||^2 = ||a||^2 + ||b||^2 - 2*(a.b)
        sum_sq = np.sum(X**2, axis=1)
        D = sum_sq[:, np.newaxis] + sum_sq[np.newaxis, :] - 2 * np.dot(X, X.T)
        D = np.maximum(D, 0.0) # Avoid negative floating point errors
        np.fill_diagonal(D, 0.0) # Self-distance is zero
        return D

    def _compute_entropy_perplexity(self, D_i, sigma):
        # Compute conditional probabilities and perplexity for a given row and standard deviation sigma
        p_i = np.exp(-D_i / (2 * sigma**2))
        p_i_sum = np.sum(p_i)
        if p_i_sum == 0:
            p_i_sum = 1e-12
        p_i = p_i / p_i_sum
        
        # Avoid log of zero
        entropy = -np.sum(p_i * np.log2(np.maximum(p_i, 1e-12)))
        return entropy, 2**entropy, p_i

    def _compute_P(self, X):
        n = X.shape[0]
        D = self._pairwise_squared_distances(X)
        P = np.zeros((n, n))
        
        # Perform binary search to find optimal sigma for each row to match target perplexity
        for i in range(n):
            D_i = np.delete(D[i], i) # Exclude self-distance
            
            low_sigma, high_sigma = 1e-10, 1e10
            sigma = 1.0
            
            # Max 50 iterations for binary search
            for _ in range(50):
                mid_sigma = (low_sigma + high_sigma) / 2.0
                entropy, perp, p_i = self._compute_entropy_perplexity(D_i, mid_sigma)
                
                if perp < self.perplexity:
                    low_sigma = mid_sigma
                else:
                    high_sigma = mid_sigma
                
                sigma = mid_sigma
                if np.abs(perp - self.perplexity) < 1e-5:
                    break
            
            # Insert neighbor probabilities back, setting self-probability to 0
            p_i_full = np.insert(p_i, i, 0.0)
            P[i] = p_i_full
        
        # Symmetrize the similarity matrix: P_ij = (P_j|i + P_i|j) / 2n
        P = (P + P.T) / (2 * n)
        # Set lower bound floor for numerical stability
        P = np.maximum(P, 1e-12)
        return P

    def _compute_Q(self, Y):
        # Compute joint probabilities Q_ij using Cauchy Student-t distribution (1 DOF)
        D_y = self._pairwise_squared_distances(Y)
        num = (1.0 + D_y) ** -1
        np.fill_diagonal(num, 0.0) # Set self-probability to 0
        
        sum_num = np.sum(num)
        Q = num / (sum_num if sum_num != 0 else 1e-12)
        Q = np.maximum(Q, 1e-12) # Safeguard log zero
        return Q, num

    def _compute_gradients(self, P, Q, Y, num):
        n = Y.shape[0]
        grad = np.zeros_like(Y)
        
        # dC/dy_i = 4 * sum_j (p_ij - q_ij) * (y_i - y_j) * (1 + ||y_i - y_j||^2)^-1
        # Vectorized implementation for speed
        PQ_diff = P - Q
        for i in range(n):
            diff = Y[i] - Y
            grad[i] = 4 * np.sum((PQ_diff[i] * num[i])[:, np.newaxis] * diff, axis=0)
        return grad

    def fit_transform(self, X):
        np.random.seed(self.random_state)
        n = X.shape[0]
        
        # Compute high-dimensional probabilities
        P = self._compute_P(X)
        
        # Apply early exaggeration (multiplier of 4) during initial steps
        P_exag = P * 4.0
        
        # Initialize low-dimensional coordinates Y using standard random normal distribution
        Y = np.random.randn(n, self.n_components) * 1e-4
        
        # Initialize velocity matrices for gradient descent momentum
        velocity = np.zeros_like(Y)
        momentum_coeff = 0.5
        
        for it in range(self.n_iter):
            P_active = P_exag if it < 100 else P
            if it == 100:
                momentum_coeff = 0.8 # Increase momentum step after early exaggeration
                
            Q, num = self._compute_Q(Y)
            grad = self._compute_gradients(P_active, Q, Y, num)
            
            # Update velocity and coordinate matrices
            velocity = momentum_coeff * velocity - self.learning_rate * grad
            Y = Y + velocity
            
            # Shift coordinates around origin to prevent drift
            Y = Y - np.mean(Y, axis=0)
            
            # Compute KL Divergence cost every 50 iterations
            if it % 50 == 0:
                cost = np.sum(P * np.log(P / Q))
                self.costs.append(cost)
                print(f"Iteration {it:4d} | KL Divergence Loss: {cost:.4f}")
                
        return Y

## 7. Running and Visualizing Our From-Scratch Solver

### WHY?
We run our from-scratch NumPy solver on a small subset of 300 samples to verify that it functions correctly and converges to a reasonable layout.

### HOW?
We slice the first 300 samples of our pre-processed dataset, run our `SimpleTSNE` solver, and plot the loss convergence curve side-by-side with the final 2D projection scatter plot.

In [ ]:
# Slice a small subset for scratch execution speed
n_scratch = 300
X_scratch = X_pca[:n_scratch]
y_scratch = y[:n_scratch]

print(f"🚀 Running From-Scratch SimpleTSNE on {n_scratch} samples...")
scratch_tsne = SimpleTSNE(n_components=2, perplexity=30.0, n_iter=300, learning_rate=100.0, random_state=42)

t0 = time.time()
Y_scratch = scratch_tsne.fit_transform(X_scratch)
t1 = time.time()

print(f"✅ Scratch solver complete! Runtime: {t1 - t0:.2f} seconds.\n")

# Plot the diagnostic dashboard side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Loss convergence curve
axes[0].plot(np.arange(0, 300, 50), scratch_tsne.costs, marker='o', color='#E91E63', linewidth=2)
axes[0].set_xlabel('Iteration Index', fontsize=11)
axes[0].set_ylabel('KL Divergence Cost', fontsize=11)
axes[0].set_title('KL Divergence Loss Convergence (Scratch t-SNE)', fontsize=13, fontweight='bold')

# Right: 2D Projected Space scatter plot
scatter = axes[1].scatter(Y_scratch[:, 0], Y_scratch[:, 1], c=y_scratch, cmap='tab10', alpha=0.8, edgecolors='black', s=50)
legend = axes[1].legend(*scatter.legend_elements(), title="Digits")
axes[1].add_artist(legend)
axes[1].set_xlabel('Component 1 (PC1)', fontsize=11)
axes[1].set_ylabel('Component 2 (PC2)', fontsize=11)
axes[1].set_title('Scratch 2D t-SNE Projection Space (300 Samples)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Library-Grade t-SNE Using Scikit-learn

### WHY?
While our manual solver is educational, scikit-learn uses the highly optimized **Barnes-Hut approximation**, which runs in $O(N \log N)$ time complexity rather than exact $O(N^2)$, making it much faster on large datasets.

### HOW?
We import `TSNE` from `sklearn.manifold`, configure it to use Barnes-Hut approximation, set perplexity to 40, and run the pipeline on our 5,000 sample dataset.

In [ ]:
print("🚀 Running Scikit-learn t-SNE (Barnes-Hut) on 5,000 samples...")
t_start = time.time()

# Configure and fit the production-grade TSNE estimator
sklearn_tsne = TSNE(
    n_components=2,
    perplexity=40.0,
    max_iter=1000,
    init='pca',
    learning_rate='auto',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
Y_lib = sklearn_tsne.fit_transform(X_pca)
t_end = time.time()

print(f"\n✅ Scikit-learn t-SNE complete!")
print(f"   Total runtime     : {t_end - t_start:.2f} seconds")
print(f"   Output shape      : {Y_lib.shape}")
print(f"   Final KL loss     : {sklearn_tsne.kl_divergence_:.4f}")

## 9. Visualizing t-SNE Embeddings

### WHY?
Visualizing the 2D projection space helps evaluate whether the dataset's classes are separated cleanly. We will plot both a standard cluster scatter plot and a density contour plot to reveal the cluster density centers.

### HOW?
We plot a 2D scatter plot colored by digit class, and a Seaborn KDE joint density contour plot showing cluster centers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: 2D Projected Space scatter plot colored by digit class
scatter = axes[0].scatter(Y_lib[:, 0], Y_lib[:, 1], c=y, cmap='tab10', alpha=0.7, edgecolors='none', s=20)
legend = axes[0].legend(*scatter.legend_elements(), title="Digits", loc='best')
axes[0].add_artist(legend)
axes[0].set_xlabel('t-SNE Component 1', fontsize=11)
axes[0].set_ylabel('t-SNE Component 2', fontsize=11)
axes[0].set_title('MNIST Digit Clusters in 2D t-SNE Projected Space', fontsize=13, fontweight='bold')

# Right: Joint Density contour plot showing cluster centers
sns.kdeplot(
    x=Y_lib[:, 0], y=Y_lib[:, 1], hue=y, palette='tab10', fill=True,
    alpha=0.4, levels=5, thresh=0.1, ax=axes[1]
)
axes[1].set_xlabel('t-SNE Component 1', fontsize=11)
axes[1].set_ylabel('t-SNE Component 2', fontsize=11)
axes[1].set_title('Cluster Density Contour Distribution (KDE)', fontsize=13, fontweight='bold')

plt.suptitle('MNIST 2D Projection Space Diagnostic Dashboard', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 10. Hyperparameter Sweeps: The Influence of Perplexity

### WHY?
Perplexity controls the "zoom level" of neighbor selection. We sweep perplexity values of 5, 40, and 100 to observe how it changes the final layout.

### HOW?
We loop through perplexity values of `[5, 40, 100]`, run `TSNE` on our PCA features, and plot the three resulting scatter plots side-by-side.

In [ ]:
perplexities = [5, 40, 100]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, perp in enumerate(perplexities):
    print(f"⚙️  Running t-SNE with perplexity={perp}...")
    tsne_sweep = TSNE(n_components=2, perplexity=perp, max_iter=500, random_state=42, n_jobs=-1)
    Y_sweep = tsne_sweep.fit_transform(X_pca)
    
    scatter = axes[idx].scatter(Y_sweep[:, 0], Y_sweep[:, 1], c=y, cmap='tab10', alpha=0.6, s=10)
    axes[idx].set_title(f"Perplexity = {perp}", fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('t-SNE Component 1', fontsize=11)
    axes[idx].set_ylabel('t-SNE Component 2', fontsize=11)

plt.suptitle('Hyperparameter Sweep: Influence of Perplexity on Clustering', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Evaluating t-SNE Quality: Trustworthiness and KNN Classifiers

### WHY?
Unlike classification models, t-SNE has no direct "correct label". We evaluate its quality using two quantitative metrics:
1. **Trustworthiness Score:** Measures how well the original neighborhood structure is preserved (Score of 1.0 is perfect).
2. **Downstream KNN Classification Accuracy:** Evaluates how cleanly separated the digit clusters are by measuring the classification accuracy of a simple 5-Nearest Neighbors classifier trained on the 2D embeddings.

### HOW?
We calculate the `trustworthiness` score on a subset of the data, fit a 5-fold cross-validated KNN classifier on our 2D t-SNE features, and plot the confusion matrix.

In [ ]:
print("📊 Evaluating t-SNE Embedding Quality...\n")

# Step 11a: Calculate Trustworthiness Score (computed on 1000 samples for speed)
score = trustworthiness(X_pca[:1000], Y_lib[:1000], n_neighbors=10)
print(f"%s Neighborhood Trustworthiness Score (n_neighbors=10): {score:.4f}" % ('✅' if score > 0.9 else '⚠️'))

# Step 11b: Train a KNN classifier on the 2D t-SNE features
knn = KNeighborsClassifier(n_neighbors=5)
cv_scores = cross_val_score(knn, Y_lib, y, cv=5, scoring='accuracy')

print(f"✅ KNN (k=5) Cross-Validation Accuracy on 2D Embeddings: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

# Step 11c: Fit KNN on split data to plot a Confusion Matrix
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(Y_lib, y, test_size=0.3, random_state=42, stratify=y)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.arange(10), yticklabels=np.arange(10))
plt.title('Confusion Matrix: KNN Classifier on t-SNE Embeddings', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.show()

# Placement & Interview Q&A

**Q1. What is the Crowding Problem?**  
**Answer:** In high dimensions, there is exponentially more "volume" around any data point than in low dimensions. Squeezing these neighbors into a 2D plane creates a space deficit, causing moderately-distant clusters to collapse into a single crowded blob.

**Q2. How does the Student t-distribution solve crowding?**  
**Answer:** The Student-t distribution has much heavier (fatter) tails than a Gaussian. For a given similarity value, the low-dimensional distance must be much larger than the corresponding high-dimensional distance, naturally pushing separate clusters apart on the 2D map.

**Q3. Why can we NOT trust global distances in a t-SNE plot?**  
**Answer:** t-SNE's KL Divergence cost function multiplies distance penalties by $p_{ij}$ (high-dimensional neighbor similarity). For far-apart points, $p_{ij} \to 0$, meaning the optimization ignores global distance relationships, making the space *between* clusters unreliable.

**Q4. Can t-SNE project new, unseen test data?**  
**Answer:** No. Standard t-SNE is non-parametric and directly optimizes the low-dimensional coordinates $Y$. It does not learn a reusable mapping function (like a projection matrix in PCA). To project new data, the optimization must be re-run on the entire combined dataset.

**Q5. Why do we typically run PCA before t-SNE?**  
**Answer:** PCA filters out high-frequency noise, speeds up pairwise distance computations, and prevents t-SNE's non-linear solver from getting trapped in poor local configurations.

---

# Key Takeaways

- **t-SNE preserves local neighborhoods.** Points that are close neighbors in the original high-dimensional space are kept close together in the 2D projection space.
- **The Cauchy Student-t distribution resolves crowding.** The fat tails of the t-distribution force separate clusters apart on the map, providing breathing room.
- **PCA pre-processing is mandatory in practice.** For high-dimensional datasets, running PCA first filters out noise and reduces execution time.
- **Hyperparameters are highly sensitive.** Perplexity acts as a neighborhood zoom level. Tuning perplexity, learning rate, and iterations is essential to avoid visual artifacts.